In [ ]:
# Import nannyml
import nannyml

# Load the US Census Employment dataset
reference, analysis, analysis_gt = nannyml.load_us_census_ma_employment_data()

# Print head of the reference data
print(reference.head())

# Print head of the analysis data
print(analysis.head())

In [ ]:
# Load the dataset
dataset_name = "green_taxi_dataset.csv"
data = pd.read_csv(dataset_name)
data.head()

In [ ]:
# Load the dataset
dataset_name = "green_taxi_dataset.csv"
data = pd.read_csv(dataset_name)
features = ['lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'fare_amount', 'pickup_time']
target = 'tip_amount'

# Split the training data
X_train = data.loc[data['partition'] == 'train', features]
y_train = data.loc[data['partition'] == 'train', target]

# Split the test data
X_test = data.loc[data['partition'] == 'test', features]
y_test = data.loc[data['partition'] == 'test', target]

# Split the prod data
X_prod = data.loc[data['partition'] == 'prod', features]
y_prod = data.loc[data['partition'] == 'prod', target]

In [ ]:
from lightgbm import LGBMRegressor

# Fit the model
model = LGBMRegressor(random_state=111, n_estimators=50, n_jobs=1)
model.fit(X_train, y_train)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Deploy the model
y_pred_prod = model.predict(X_prod)

# Create reference and analysis set
reference = X_test.copy() # Copy test set features
reference['y_pred'] = y_pred_test # Add models predictions on test set
reference['tip_amount'] = y_test # Add labels(ground truth)
reference = reference.join(data['lpep_pickup_datetime']) # Add timestamp column

analysis = X_prod.copy() # Copy production set features
analysis['y_pred'] = y_pred_prod # Add models predictions on production set
analysis = analysis.join(data['lpep_pickup_datetime']) # Add timestamp column

In [ ]:
estimator nannyml.CBPE(...)
estimator.fit(reference)
results = estimator.estimate (analysis)
figure results.plot()
figure.show()

In [ ]:
estimator = nannyml.DLE(y_pred='y_pred',
    timestamp_column_name='lpep_pickup_datetime',
    feature_column_names=features,
    chunk_period='d',
    y_true='tip_amount',
    metrics=['mse'])

# Fit the reference data to the DLE algorithm
estimator.fit(reference)

# Estimate the performance on the analysis data
results = estimator.estimate(analysis)

# Plot and show the results
results.plot().show()

In [ ]:
# Intialize the calculator
calculator = nannyml.PerformanceCalculator(
    y_true='tip_amount',
    y_pred='y_pred',
    chunk_period='d',
  	metrics=['mae'],
    timestamp_column_name='lpep_pickup_datetime',
    problem_type='regression')

# Fit the calculator
calculator.fit(reference)
realized_results = calculator.calculate(analysis)

# Show comparison plot for realized and estimated performance
realized_results.compare(estimated_results).plot().show()

In [ ]:
reference, analysis, analysis_gt = nannyml.load_us_census_ma_employment_data()

# Initialize the CBPE algorithm
cbpe = nannyml.CBPE(
    y_pred_proba='predicted_probability',
    y_pred='prediction',
    y_true='employed',
    metrics = ['roc_auc', 'accuracy'],
    problem_type = 'classification_binary',
    chunk_size = 5000,
)

cbpe = cbpe.fit(reference)
estimated_results = cbpe.estimate(analysis)
estimated_results.plot().show()

In [ ]:
reference, analysis, analysis_gt = nannyml.load_us_census_ma_employment_data()

# Initialize the CBPE algorithm
cbpe = nannyml.CBPE(
    y_pred_proba='predicted_probability',
    y_pred='prediction',
    y_true='employed',
    metrics = ['roc_auc', 'accuracy', 'f1'],
    problem_type = 'classification_binary',
    chunk_number = 8,
)

cbpe = cbpe.fit(reference)
estimated_results = cbpe.estimate(analysis)
estimated_results.plot().show()

In [ ]:
# Import custom thresholds
from nannyml.thresholds import StandardDeviationThreshold, ConstantThreshold

# Initialize custom thresholds
stdt = StandardDeviationThreshold(std_lower_multiplier=2, std_upper_multiplier=2)
ct = ConstantThreshold(lower=0.9, upper=0.98)

# Initialize the CBPE algorithm
estimator = nannyml.CBPE(
    problem_type='classification_binary',
    y_pred_proba='predicted_probability',
    y_pred='prediction',
    y_true='employed',
    metrics=['roc_auc', 'accuracy', 'f1'],
    thresholds={'f1': ct, 'accuracy' : stdt})

In [ ]:
# Filter estimated results for the roc_auc metric and convert them to a dataframe
display(estimated_results.filter(metrics=['roc_auc']).to_df())

# Filter estimated results for the reference period and convert them to a dataframe
display(estimated_results.filter(period='reference').to_df())

# Filter the estimated results for the accuracy metric
display(estimated_results.filter(metrics=['accuracy']).plot().show())

# Filter the estimated results for the analysis period, as well as for accuracy and roc_auc metrics
display(estimated_results.filter(period='analysis', metrics=['accuracy', 'roc_auc']).plot().show())

In [ ]:
# Custom business value thresholds
ct = ConstantThreshold(lower=0, upper=150000)
# Intialize the performance calculator
calc = PerformanceCalculator(problem_type='classification_binary',
			y_pred_proba='y_pred_proba',
  			timestamp_column_name="timestamp", 		
  			y_pred='y_pred',
  			y_true='is_canceled',
            chunk_period='m',
  			metrics=['business_value', 'roc_auc'],
  			business_value_matrix = [[0, -100],[-200, 1500]],
  			thresholds={"business_value": ct})
calc = calc.fit(reference)
calc_res = calc.calculate(analysis)
calc_res.filter(period='analysis').plot().show()

In [ ]:
# Create standard deviation thresholds
stdt = StandardDeviationThreshold(std_lower_multiplier=2, std_upper_multiplier=1)

# Define feature columns
feature_column_names = ["country", "lead_time", "parking_spaces", "hotel"]

# Intialize, fit, and show results of multivariate drift calculator
mv_calc = nannyml.DataReconstructionDriftCalculator(
    column_names=feature_column_names,
	threshold = stdt,
    timestamp_column_name='timestamp',
    chunk_period='m')
mv_calc.fit(reference)
mv_results = mv_calc.calculate(analysis)
mv_results.filter(period='analysis').compare(perf_results).plot().show()

In [ ]:
# Intialize the univariate drift calculator
uv_calc = nannyml.UnivariateDriftCalculator(
    column_names=feature_column_names,
    timestamp_column_name='timestamp',
    chunk_period='m',
    continuous_methods=['wasserstein', 'jensen_shannon'],
    categorical_methods=['l_infinity', 'chi2'],
)

# Plot the results
uv_calc.fit(reference)
uv_results = uv_calc.calculate(analysis)
uv_results.plot().show()

In [ ]:
# Initialize the alert count ranker
alert_count_ranker = nannyml.AlertCountRanker()
alert_count_ranked_features = alert_count_ranker.rank(
    uv_results.filter(methods=['wasserstein', 'l_infinity']))

display(alert_count_ranked_features)

# Initialize the correlation ranker
correlation_ranker = nannyml.CorrelationRanker()
correlation_ranker.fit(perf_results.filter(period='reference'))

correlation_ranked_features = correlation_ranker.rank(
    uv_results.filter(methods=['wasserstein', 'l_infinity']),
    perf_results)
display(correlation_ranked_features)

In [ ]:
# Filter and create drift plots
drift_results = uv_results.filter(
    period='analysis',
    column_names=['hotel', 'country']
    ).plot(kind='drift')

# Filter and create distribution plots
distribution_results = uv_results.filter(
    period='analysis',
    column_names=['hotel', 'country']
    ).plot(kind='distribution')

# Show the plots
drift_results.show()
distribution_results.show()

In [ ]:
# Define analyzed columns
selected_columns = ['country', 'lead_time', 'parking_spaces', 'hotel']

# Intialize missing values calculator
ms_calc = nannyml.MissingValuesCalculator(
    column_names=selected_columns,
    chunk_period='m',
    timestamp_column_name='timestamp'
)

# Fit, calculate and plot the results
ms_calc.fit(reference)
ms_results = ms_calc.calculate(analysis)
ms_results.plot().show()

In [ ]:
# Define analyzed categorical columns
categorical_columns = ['country', 'hotel']

# Intialize unseen values calculator
us_calc = nannyml.UnseenValuesCalculator(
  	column_names=categorical_columns, 
  	chunk_period='m', 
  	timestamp_column_name='timestamp'
)

# Fit, calculate and plot the results
us_calc.fit(reference)
us_results = us_calc.calculate(analysis)
us_results.filter(period='analysis').plot().show()

In [ ]:
# Define analyzed column
analyzed_column = ['lead_time']

# Intialize sum values calculator
sum_calc = nannyml.SummaryStatsSumCalculator(
    column_names=analyzed_column, 
    chunk_period='m', 
    timestamp_column_name='timestamp'
)

sum_calc.fit(reference)
sum_calc_res = sum_calc.calculate(analysis)
sum_calc_res.plot().show()

In [ ]:
analyzed_column = ['lead_time']

# Intialize median values calculator
med_calc = nannyml.SummaryStatsMedianCalculator(
    column_names=analyzed_column, 
    chunk_period='m', 
    timestamp_column_name='timestamp'
)

med_calc.fit(reference)
med_calc_res = med_calc.calculate(analysis)

# Filter the results
med_calc_res.filter(period="analysis").plot().show()

In [ ]:
# Define analyzed column
analyzed_column = ['lead_time']

# Intialize standard deviation values calculator
std_calc = nannyml.SummaryStatsStdCalculator(
    column_names=analyzed_column, 
    chunk_period='m', 
    timestamp_column_name='timestamp'
)

# Fit, calculate and plot the results
std_calc.fit(reference)
std_calc_res = std_calc.calculate(analysis)
std_calc_res.filter(period="analysis").plot().show()

In [ ]:
# Estimate the performance
estimator.fit(reference)
estimated_results = estimator.estimate(analysis)
estimated_results.plot().show()

In [ ]:
estimator.fit(reference)
estimated_results = estimator.estimate(analysis)

# Calculate multivariate drift
mv_calc = nannyml.DataReconstructionDriftCalculator(column_names=features, chunk_size=5000)
mv_calc.fit(reference)
mv_results = mv_calc.calculate(analysis)
display(mv_results.filter(period="analysis").compare(estimated_results).plot().show())

In [ ]:
estimator.fit(reference)
estimated_performance = estimator.estimate(analysis)
mv_calc = nannyml.DataReconstructionDriftCalculator(column_names=features, chunk_size=5000)
mv_calc.fit(reference)
mv_results = mv_calc.calculate(analysis)

# Calculate univariate drift
uv_calculator.fit(reference)
uv_results = uv_calculator.calculate(analysis)

# Check the most drifting features
alert_count_ranked_features = alert_count_ranker.rank(uv_results)
display(alert_count_ranked_features.head())